In [1]:
import numpy as np
import pandas as pd

# **Performing database-style operations on DataFrames** 

In [2]:
weather_df = pd.read_csv('../Datasets/nyc_weather_2018.csv')

weather_df.head()

,date,datatype,station,attributes,value
0,2018-01-01T00:00:00,PRCP,GHCND:US1CTFR0039,",,N,",0.0
1,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0015,",,N,",0.0
2,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0015,",,N,",0.0
3,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0017,",,N,",0.0
4,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0017,",,N,",0.0


## **Querying DataFrames**

In [3]:
snow_data = weather_df.query(
    'datatype == "SNOW" and value > 0'
    'and station.str.contains("US1NY")'
)

snow_data.head()

,date,datatype,station,attributes,value
114,2018-01-01T00:00:00,SNOW,GHCND:US1NYWC0019,",,N,",25.0
789,2018-01-04T00:00:00,SNOW,GHCND:US1NYNS0007,",,N,",41.0
794,2018-01-04T00:00:00,SNOW,GHCND:US1NYNS0018,",,N,",10.0
798,2018-01-04T00:00:00,SNOW,GHCND:US1NYNS0024,",,N,",89.0
800,2018-01-04T00:00:00,SNOW,GHCND:US1NYNS0030,",,N,",102.0


In [6]:
weather_df[
    (weather_df['datatype'] == "SNOW") & (weather_df['value'] > 0) & (weather_df['station'].str.contains("US1NY"))
].equals(snow_data)

True

> **Tip :** When using Boolean logic with the **`query()`** method, we can use both logical operators (and, or, not) and bitwise operators (&, |, ~).

## **Merging DataFrames**

![Understanding join types](../Images/Understanding_join_types.png)

In [7]:
station_info = pd.read_csv('../Datasets/weather_stations.csv')

station_info.head()

,id,name,latitude,longitude,elevation
0,GHCND:US1CTFR0022,"STAMFORD 2.6 SSW, CT US",41.064100,-73.577000,36.6
1,GHCND:US1CTFR0039,"STAMFORD 4.2 S, CT US",41.037788,-73.568176,6.4
2,GHCND:US1NJBG0001,"BERGENFIELD 0.3 SW, NJ US",40.921298,-74.001983,20.1
3,GHCND:US1NJBG0002,"SADDLE BROOK TWP 0.6 E, NJ US",40.902694,-74.083358,16.8
4,GHCND:US1NJBG0003,"TENAFLY 1.3 W, NJ US",40.914670,-73.977500,21.6


In [8]:
station_info['id'].describe()

count                   279
unique                  279
top       GHCND:US1CTFR0022
freq                      1
Name: id, dtype: object

In [9]:
weather_df['station'].describe()

count                 78780
unique                  110
top       GHCND:USW00094789
freq                   4270
Name: station, dtype: object

In [10]:
weather_df.shape[0], station_info.shape[0]

(78780, 279)

In [11]:
def get_rows_count(*dfs):
    return [df.shape[0] for df in dfs]

In [12]:
get_rows_count(weather_df, station_info)

[78780, 279]

In [13]:
inner_join = weather_df.merge(
    station_info, left_on='station', right_on='id'
)

inner_join.sample(n=5, random_state=0)

,date,datatype,station,attributes,value,id,name,latitude,longitude,elevation
10739,2018-02-17T00:00:00,PRCP,GHCND:USC00066655,",,7,0700",4.1,GHCND:USC00066655,"PUTNAM LAKE, CT US",41.082500,-73.638600,91.4
45188,2018-07-27T00:00:00,SNOW,GHCND:US1NJES0019,",,N,",0.0,GHCND:US1NJES0019,"WEST CALDWELL TWP 1.3 NE, NJ US",40.861500,-74.277500,81.4
59823,2018-10-05T00:00:00,PRCP,GHCND:US1NJES0024,",,N,",0.0,GHCND:US1NJES0024,"CEDAR GROVE TWP 0.4 W, NJ US",40.855695,-74.235564,108.5
10852,2018-02-17T00:00:00,TMIN,GHCND:USW00094789,",,W,2400",-2.1,GHCND:USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",40.639150,-73.764010,3.4
46755,2018-08-03T00:00:00,AWND,GHCND:USW00094745,",,W,",1.8,GHCND:USW00094745,"WESTCHESTER CO AIRPORT, NY US",41.062360,-73.704630,111.9


In [14]:
weather_df.merge(
    station_info.rename(dict(id='station'), axis=1),
    on= 'station'
).sample(n=5, random_state=0)

,date,datatype,station,attributes,value,name,latitude,longitude,elevation
10739,2018-02-17T00:00:00,PRCP,GHCND:USC00066655,",,7,0700",4.1,"PUTNAM LAKE, CT US",41.082500,-73.638600,91.4
45188,2018-07-27T00:00:00,SNOW,GHCND:US1NJES0019,",,N,",0.0,"WEST CALDWELL TWP 1.3 NE, NJ US",40.861500,-74.277500,81.4
59823,2018-10-05T00:00:00,PRCP,GHCND:US1NJES0024,",,N,",0.0,"CEDAR GROVE TWP 0.4 W, NJ US",40.855695,-74.235564,108.5
10852,2018-02-17T00:00:00,TMIN,GHCND:USW00094789,",,W,2400",-2.1,"JFK INTERNATIONAL AIRPORT, NY US",40.639150,-73.764010,3.4
46755,2018-08-03T00:00:00,AWND,GHCND:USW00094745,",,W,",1.8,"WESTCHESTER CO AIRPORT, NY US",41.062360,-73.704630,111.9


> **Tip :** We can join on multiple columns by passing the list of column names to the on parameter or to the **`left_on`** and **`right_on`** parameters.

In [15]:
left_join = station_info.merge(
    weather_df, left_on='id', right_on='station', how='left'
)

right_join = weather_df.merge(
    station_info, left_on='station', right_on='id', how='right'
)

In [31]:
right_join[right_join.datatype.isna()].head()

,date,datatype,station,attributes,value,id,name,latitude,longitude,elevation
0,NaN,NaN,NaN,NaN,NaN,GHCND:US1CTFR0022,"STAMFORD 2.6 SSW, CT US",41.064100,-73.577000,36.6
344,NaN,NaN,NaN,NaN,NaN,GHCND:US1NJBG0001,"BERGENFIELD 0.3 SW, NJ US",40.921298,-74.001983,20.1
345,NaN,NaN,NaN,NaN,NaN,GHCND:US1NJBG0002,"SADDLE BROOK TWP 0.6 E, NJ US",40.902694,-74.083358,16.8
718,NaN,NaN,NaN,NaN,NaN,GHCND:US1NJBG0005,"WESTWOOD 0.8 ESE, NJ US",40.983041,-74.015858,15.8
719,NaN,NaN,NaN,NaN,NaN,GHCND:US1NJBG0006,"RAMSEY 0.6 E, NJ US",41.058611,-74.134068,112.2


In [36]:
left_join.sort_index(axis=1).sort_values(['date', 'station'], ignore_index=True).equals(
    right_join.sort_index(axis=1).sort_values(['date', 'station'], ignore_index=True)
)

True

In [37]:
get_rows_count(inner_join, left_join, right_join)

[78780, 78949, 78949]

In [38]:
outer_join = weather_df.merge(
    station_info[station_info.id.str.contains('US1NY')],
    left_on='station', right_on='id', how='outer', indicator=True
)

outer_join.head()

,date,datatype,station,attributes,value,id,name,latitude,longitude,elevation,_merge
0,2018-01-01T00:00:00,PRCP,GHCND:US1CTFR0039,",,N,",0.0,NaN,NaN,NaN,NaN,NaN,left_only
1,2018-01-02T00:00:00,PRCP,GHCND:US1CTFR0039,",,N,",0.0,NaN,NaN,NaN,NaN,NaN,left_only
2,2018-01-03T00:00:00,PRCP,GHCND:US1CTFR0039,",,N,",0.0,NaN,NaN,NaN,NaN,NaN,left_only
3,2018-01-05T00:00:00,DAPR,GHCND:US1CTFR0039,",,N,",2.0,NaN,NaN,NaN,NaN,NaN,left_only
4,2018-01-05T00:00:00,MDPR,GHCND:US1CTFR0039,",,N,",15.5,NaN,NaN,NaN,NaN,NaN,left_only


In [42]:
pd.concat(
    [outer_join.query(f'_merge == "{tmerge}"')
        .sample(n=2, random_state=0)
    for tmerge in outer_join._merge.unique()
]).sort_index()

,date,datatype,station,attributes,value,id,name,latitude,longitude,elevation,_merge
28719,2018-03-05T00:00:00,PRCP,GHCND:US1NYNS0037,",,N,",0.3,GHCND:US1NYNS0037,"WANTAGH 1.1 NNE, NY US",40.683311,-73.503837,10.4,both
30828,2018-12-19T00:00:00,PRCP,GHCND:US1NYNS0046,",,N,",0.0,GHCND:US1NYNS0046,"MASSAPEQUA PARK 1.2 N, NY US",40.698077,-73.449893,10.7,both
32004,NaN,NaN,NaN,NaN,NaN,GHCND:US1NYQN0033,"HOWARD BEACH 0.4 NNW, NY US",40.662099,-73.841345,2.1,right_only
34194,NaN,NaN,NaN,NaN,NaN,GHCND:US1NYWC0009,"NEW ROCHELLE 1.3 S, NY US",40.904000,-73.777000,21.9,right_only
62899,2018-07-21T00:00:00,TMAX,GHCND:USW00054787,",,W,",24.4,NaN,NaN,NaN,NaN,NaN,left_only
73018,2018-07-19T00:00:00,TMAX,GHCND:USW00094745,",,W,2400",27.2,NaN,NaN,NaN,NaN,NaN,left_only


In [43]:
dirty_data = pd.read_csv(
    '../Datasets/dirty_data.csv',
    index_col='date'
).drop_duplicates().drop(columns='SNWD')

dirty_data.head()

,station,PRCP,SNOW,TMAX,TMIN,TOBS,WESF,inclement_weather
date,,,,,,,,
2018-01-01T00:00:00,?,0.0,0.0,5505.0,-40.0,NaN,NaN,NaN
2018-01-02T00:00:00,GHCND:USC00280907,0.0,0.0,-8.3,-16.1,-12.2,NaN,False
2018-01-03T00:00:00,GHCND:USC00280907,0.0,0.0,-4.4,-13.9,-13.3,NaN,False
2018-01-04T00:00:00,?,20.6,229.0,5505.0,-40.0,NaN,19.3,True
2018-01-05T00:00:00,?,0.3,NaN,5505.0,-40.0,NaN,NaN,NaN


In [44]:
valid_station = dirty_data.query(
    'station != "?"'
).drop(columns=['WESF', 'station'])

station_with_wesf = dirty_data.query(
    'station == "?"'
).drop(columns=['station', 'TOBS', 'TMIN', 'TMAX'])

In [45]:
valid_station.merge(
    station_with_wesf, how='left', left_index=True, right_index=True
).query('WESF > 0').head()

,PRCP_x,SNOW_x,TMAX,TMIN,TOBS,inclement_weather_x,PRCP_y,SNOW_y,WESF,inclement_weather_y
date,,,,,,,,,,
2018-01-30T00:00:00,0.0,0.0,6.7,-1.7,-0.6,False,1.5,13.0,1.8,True
2018-03-08T00:00:00,48.8,NaN,1.1,-0.6,1.1,False,28.4,NaN,28.7,NaN
2018-03-13T00:00:00,4.1,51.0,5.6,-3.9,0.0,True,3.0,13.0,3.0,True
2018-03-21T00:00:00,0.0,0.0,2.8,-2.8,0.6,False,6.6,114.0,8.6,True
2018-04-02T00:00:00,9.1,127.0,12.8,-1.1,-1.1,True,14.0,152.0,15.2,True


In [46]:
valid_station.merge(
    station_with_wesf, 
    how='left', 
    left_index=True, 
    right_index=True,
    suffixes=('', '_?')
).query('WESF > 0').head()

,PRCP,SNOW,TMAX,TMIN,TOBS,inclement_weather,PRCP_?,SNOW_?,WESF,inclement_weather_?
date,,,,,,,,,,
2018-01-30T00:00:00,0.0,0.0,6.7,-1.7,-0.6,False,1.5,13.0,1.8,True
2018-03-08T00:00:00,48.8,NaN,1.1,-0.6,1.1,False,28.4,NaN,28.7,NaN
2018-03-13T00:00:00,4.1,51.0,5.6,-3.9,0.0,True,3.0,13.0,3.0,True
2018-03-21T00:00:00,0.0,0.0,2.8,-2.8,0.6,False,6.6,114.0,8.6,True
2018-04-02T00:00:00,9.1,127.0,12.8,-1.1,-1.1,True,14.0,152.0,15.2,True


In [47]:
# The join() method will always use the index of the left dataframe to join, but it can use a column in the right dataframe 
#if its name is passed to the on parameter.

valid_station.join(
    station_with_wesf,
    how='left',
    rsuffix='_?',
).query('WESF > 0').head()

,PRCP,SNOW,TMAX,TMIN,TOBS,inclement_weather,PRCP_?,SNOW_?,WESF,inclement_weather_?
date,,,,,,,,,,
2018-01-30T00:00:00,0.0,0.0,6.7,-1.7,-0.6,False,1.5,13.0,1.8,True
2018-03-08T00:00:00,48.8,NaN,1.1,-0.6,1.1,False,28.4,NaN,28.7,NaN
2018-03-13T00:00:00,4.1,51.0,5.6,-3.9,0.0,True,3.0,13.0,3.0,True
2018-03-21T00:00:00,0.0,0.0,2.8,-2.8,0.6,False,6.6,114.0,8.6,True
2018-04-02T00:00:00,9.1,127.0,12.8,-1.1,-1.1,True,14.0,152.0,15.2,True


![Set operations](../Images/Set_operations.png)

In [48]:
weather_df.set_index('station', inplace=True)
station_info.set_index('id', inplace=True)

In [49]:
weather_df.index.intersection(station_info.index)

Index(['GHCND:US1CTFR0039', 'GHCND:US1NJBG0015', 'GHCND:US1NJBG0017',
       'GHCND:US1NJBG0018', 'GHCND:US1NJBG0023', 'GHCND:US1NJBG0030',
       'GHCND:US1NJBG0039', 'GHCND:US1NJBG0044', 'GHCND:US1NJES0018',
       'GHCND:US1NJES0024',
       ...
       'GHCND:US1NJBG0037', 'GHCND:USC00284987', 'GHCND:US1NJES0031',
       'GHCND:US1NJES0029', 'GHCND:US1NJMD0086', 'GHCND:US1NJMS0097',
       'GHCND:US1NJMN0081', 'GHCND:US1NJMD0088', 'GHCND:US1NJES0040',
       'GHCND:US1NYQN0029'],
      dtype='object', length=110)

In [50]:
weather_df.index.difference(station_info.index)

Index([], dtype='object')

In [51]:
station_info.index.difference(weather_df.index)

Index(['GHCND:US1CTFR0022', 'GHCND:US1NJBG0001', 'GHCND:US1NJBG0002',
       'GHCND:US1NJBG0005', 'GHCND:US1NJBG0006', 'GHCND:US1NJBG0008',
       'GHCND:US1NJBG0011', 'GHCND:US1NJBG0012', 'GHCND:US1NJBG0013',
       'GHCND:US1NJBG0020',
       ...
       'GHCND:USC00308322', 'GHCND:USC00308749', 'GHCND:USC00308946',
       'GHCND:USC00309117', 'GHCND:USC00309270', 'GHCND:USC00309400',
       'GHCND:USC00309466', 'GHCND:USC00309576', 'GHCND:USW00014708',
       'GHCND:USW00014786'],
      dtype='object', length=169)

> **Tip :** We can use the **`symmetric_difference()`** method on the indices of the dataframes involved in the join to see what will be lost from both sides: **`index_1.symmetric_difference(index_2)`**. The result will be the values that are only in one of the indices.

In [52]:
weather_df.index.unique().union(station_info.index)

Index(['GHCND:US1CTFR0022', 'GHCND:US1CTFR0039', 'GHCND:US1NJBG0001',
       'GHCND:US1NJBG0002', 'GHCND:US1NJBG0003', 'GHCND:US1NJBG0005',
       'GHCND:US1NJBG0006', 'GHCND:US1NJBG0008', 'GHCND:US1NJBG0010',
       'GHCND:US1NJBG0011',
       ...
       'GHCND:USW00014708', 'GHCND:USW00014732', 'GHCND:USW00014734',
       'GHCND:USW00014786', 'GHCND:USW00054743', 'GHCND:USW00054787',
       'GHCND:USW00094728', 'GHCND:USW00094741', 'GHCND:USW00094745',
       'GHCND:USW00094789'],
      dtype='object', length=279)

# **Using DataFrame operations to enrich data**

In [2]:
weather_df = pd.read_csv(
    '../Datasets/nyc_weather_2018.csv',
    parse_dates=['date']
)

fb_df = pd.read_csv(
    '../Datasets/fb_2018.csv',
    index_col='date',
    parse_dates=True
)

## **Arithmetic and statistics**

In [55]:
fb_df.assign(
    abs_z_score_volume = lambda x: x.volume.sub(x.volume.mean()).div(x.volume.std()).abs()
).query('abs_z_score_volume > 3')

,open,high,low,close,volume,abs_z_score_volume
date,,,,,,
2018-03-19,177.01,177.17,170.06,172.56,88140060,3.145078
2018-03-20,167.47,170.20,161.95,168.15,129851768,5.315169
2018-03-21,164.80,173.40,163.30,169.39,106598834,4.105413
2018-03-26,160.82,161.10,149.02,160.06,126116634,5.120845
2018-07-26,174.89,180.13,173.75,176.26,169803668,7.393705


In [57]:
fb_df.assign(
    volume_pct_change= fb_df.volume.pct_change(),
    volume_pct_rank= lambda x: x.volume_pct_change.abs().rank(ascending=False)
).nsmallest(5, 'volume_pct_rank')

,open,high,low,close,volume,volume_pct_change,volume_pct_rank
date,,,,,,,
2018-01-12,178.06,181.48,177.40,179.37,77551299,7.087876,1.0
2018-03-19,177.01,177.17,170.06,172.56,88140060,2.611789,2.0
2018-07-26,174.89,180.13,173.75,176.26,169803668,1.628841,3.0
2018-09-21,166.64,167.25,162.81,162.93,45994800,1.428956,4.0
2018-03-26,160.82,161.10,149.02,160.06,126116634,1.352496,5.0


In [58]:
fb_df['2018-01-11': '2018-01-12']

,open,high,low,close,volume
date,,,,,
2018-01-11,188.40,188.40,187.38,187.77,9588587
2018-01-12,178.06,181.48,177.40,179.37,77551299


In [62]:
(fb_df > 215).any()

open       True
high       True
low       False
close      True
volume     True
dtype: bool

In [63]:
(fb_df > 215).all()

open      False
high      False
low       False
close     False
volume     True
dtype: bool

## **Binning**

> **Important note :** While binning our data can make certain parts of the analysis easier, keep in mind that it will reduce the information in that field since the granularity is reduced.

In [67]:
(fb_df.volume.value_counts() > 1).sum()

np.int64(0)

In [68]:
(fb_df.volume.value_counts() > 1).any()

np.False_

In [71]:
volume_binned = pd.cut(
    fb_df.volume, bins=3, labels=['low', 'meduim', 'high']
)

volume_binned.value_counts()

volume
low       240
meduim      8
high        3
Name: count, dtype: int64

> **Tip :** Note that we provided labels for each bin here; if we don't do this, each bin will be labeled by the interval of values it includes, which may or may not be helpful for us, depending on our application. If we want to both label the values and see the bins afterward, we can pass in retbins=True when we call **`pd.cut()`**. Then, we can access the binned data as the first element of the tuple that is returned, and the bin ranges themselves as the second element.

In [73]:
fb_df[volume_binned == 'high'].sort_values('volume', ascending=False)

,open,high,low,close,volume
date,,,,,
2018-07-26,174.89,180.13,173.75,176.26,169803668
2018-03-20,167.47,170.20,161.95,168.15,129851768
2018-03-26,160.82,161.10,149.02,160.06,126116634


In [74]:
fb_df['2018-07-25':'2018-07-26']

,open,high,low,close,volume
date,,,,,
2018-07-25,215.715,218.62,214.27,217.50,64592585
2018-07-26,174.890,180.13,173.75,176.26,169803668


In [75]:
fb_df['2018-03-16':'2018-03-20']

,open,high,low,close,volume
date,,,,,
2018-03-16,184.49,185.33,183.41,185.09,24403438
2018-03-19,177.01,177.17,170.06,172.56,88140060
2018-03-20,167.47,170.20,161.95,168.15,129851768


In [77]:
volume_qbinned = pd.qcut(
    fb_df.volume, q=4, labels=["Q1", "Q2", "Q3", "Q4"]
)

volume_qbinned.value_counts()

volume
Q1    63
Q2    63
Q4    63
Q3    62
Name: count, dtype: int64

> **Tip :** In both of these examples, we let pandas calculate the bin ranges; however, both **`pd.cut()`** and **`pd.qcut()`** allow us to specify the upper bounds for each bin as a list.

## **Applying functions**

In [3]:
central_park_weather = weather_df.query(
    'station == "GHCND:USW00094728"'
).pivot(index='date', columns='datatype', values='value')

In [4]:
oct_weather_z_scores = central_park_weather.loc['2018-10', ['TMIN', 'TMAX', 'PRCP']] \
    .apply(lambda x: x.sub(x.mean()).div(x.std()))

oct_weather_z_scores.describe().T

,count,mean,std,min,25%,50%,75%,max
datatype,,,,,,,,
TMIN,31.0,-1.790682e-16,1.0,-1.339112,-0.751019,-0.474269,1.065152,1.843511
TMAX,31.0,1.951844e-16,1.0,-1.305582,-0.870013,-0.138258,1.011643,1.604016
PRCP,31.0,1.038596e-16,1.0,-0.394438,-0.394438,-0.394438,-0.240253,3.936167


In [86]:
oct_weather_z_scores.query('PRCP > 3').PRCP

date
2018-10-27    3.936167
Name: PRCP, dtype: float64

In [87]:
central_park_weather.loc['2018-10', 'PRCP'].describe()

count    31.000000
mean      2.941935
std       7.458542
min       0.000000
25%       0.000000
50%       0.000000
75%       1.150000
max      32.300000
Name: PRCP, dtype: float64

> **Note :** that there is also an **`applymap()`** method if the function we want to apply isn't vectorized. Alternatively, we can use **`np.vectorize()`** to vectorize our functions for use with **`apply()`**.

## **Window calculations**

### **Rolling windows**

In [15]:
central_park_weather.loc['2018-10'].assign(
    rolling_PRCP = lambda x: x.PRCP.rolling('3D').sum() # performing the rolling 3-day sum, each date will show the sum of that day's 
)[['PRCP', 'rolling_PRCP']].head(7).T                   # and the previous two days' precipitation

date,2018-10-01,2018-10-02,2018-10-03,2018-10-04,2018-10-05,2018-10-06,2018-10-07
datatype,,,,,,,
PRCP,0.0,17.5,0.0,1.0,0.0,0.0,0.0
rolling_PRCP,0.0,17.5,17.5,18.5,1.0,1.0,0.0


> **Tip :** If we want to use dates for the rolling calculation, but don't have dates in the index, we can pass the name of our date column to the on parameter in the call to **`rolling()`**. Conversely, if we want to use an integer index of row numbers, we can simply pass in an integer as the window; for example, **`rolling(3)`** for a 3-row window.

In [16]:
central_park_weather.loc['2018-10']\
    .rolling('3D')\
        .mean()\
            .head(7) \
                .iloc[:, :6]

datatype,AWND,PRCP,SNOW,SNWD,TMAX,TMIN
date,,,,,,
2018-10-01,0.900000,0.000000,0.0,0.0,24.400000,17.200000
2018-10-02,0.900000,8.750000,0.0,0.0,24.700000,17.750000
2018-10-03,0.966667,5.833333,0.0,0.0,24.233333,17.566667
2018-10-04,0.800000,6.166667,0.0,0.0,24.233333,17.200000
2018-10-05,1.033333,0.333333,0.0,0.0,23.133333,16.300000
2018-10-06,0.833333,0.333333,0.0,0.0,22.033333,16.300000
2018-10-07,1.066667,0.000000,0.0,0.0,22.600000,17.400000


In [21]:
central_park_weather['2018-10-01':'2018-10-07'].rolling('3D').agg({
    'TMAX': 'max', 'TMIN': 'min',
    'AWND': 'mean', 'PRCP': 'sum'
}).join( # join with original data for comparison
    central_park_weather[['TMAX', 'TMIN', 'AWND', 'PRCP']],
    lsuffix='_rolling'
).sort_index(axis=1)

datatype,AWND,AWND_rolling,PRCP,PRCP_rolling,TMAX,TMAX_rolling,TMIN,TMIN_rolling
date,,,,,,,,
2018-10-01,0.9,0.900000,0.0,0.0,24.4,24.4,17.2,17.2
2018-10-02,0.9,0.900000,17.5,17.5,25.0,25.0,18.3,17.2
2018-10-03,1.1,0.966667,0.0,17.5,23.3,25.0,17.2,17.2
2018-10-04,0.4,0.800000,1.0,18.5,24.4,25.0,16.1,16.1
2018-10-05,1.6,1.033333,0.0,1.0,21.7,24.4,15.6,15.6
2018-10-06,0.5,0.833333,0.0,1.0,20.0,24.4,17.2,15.6
2018-10-07,1.1,1.066667,0.0,0.0,26.1,26.1,19.4,15.6


> **Tip :** We can also use **`variable-width`** windows with a little extra effort: we can either create a subclass of **`BaseIndexer`** and provide the logic for determining the window bounds in the **`get_window_bounds()`** method (more information can be found at https://pandas.pydata.org/pandas-docs/stable/user_guide/computation.html#custom-window-rolling), or we can use one of the predefined classes in the **`pandas.api.indexers`** module. The notebook we are currently working in contains an example of using the **`VariableOffsetWindowIndexer`** class to perform a 3-business day rolling calculation.

### **Expanding windows**

In [25]:
central_park_weather.loc['2018-06'].assign(
    TOTAL_PRCP = lambda x: x.PRCP.cumsum(),
    AVG_PRCP = lambda x: x.PRCP.expanding().mean()
).head(10)[['PRCP', 'TOTAL_PRCP', 'AVG_PRCP']].T

date,2018-06-01,2018-06-02,2018-06-03,2018-06-04,2018-06-05,2018-06-06,2018-06-07,2018-06-08,2018-06-09,2018-06-10
datatype,,,,,,,,,,
PRCP,6.9,2.00,6.4,4.10,0.00,0.000000,0.000000,0.000,0.000000,0.30
TOTAL_PRCP,6.9,8.90,15.3,19.40,19.40,19.400000,19.400000,19.400,19.400000,19.70
AVG_PRCP,6.9,4.45,5.1,4.85,3.88,3.233333,2.771429,2.425,2.155556,1.97


In [29]:
central_park_weather['2018-10-01':'2018-10-07'].expanding().agg({
    'TMAX': 'max', 'TMIN': 'min',
    'AWND': 'mean', 'PRCP': 'sum'
}).join(
    central_park_weather[['TMAX', 'TMIN', 'AWND', 'PRCP']],
    lsuffix = '_expanding'
).sort_index(axis=1)

datatype,AWND,AWND_expanding,PRCP,PRCP_expanding,TMAX,TMAX_expanding,TMIN,TMIN_expanding
date,,,,,,,,
2018-10-01,0.9,0.900000,0.0,0.0,24.4,24.4,17.2,17.2
2018-10-02,0.9,0.900000,17.5,17.5,25.0,25.0,18.3,17.2
2018-10-03,1.1,0.966667,0.0,17.5,23.3,25.0,17.2,17.2
2018-10-04,0.4,0.825000,1.0,18.5,24.4,25.0,16.1,16.1
2018-10-05,1.6,0.980000,0.0,18.5,21.7,25.0,15.6,15.6
2018-10-06,0.5,0.900000,0.0,18.5,20.0,25.0,17.2,15.6
2018-10-07,1.1,0.928571,0.0,18.5,26.1,26.1,19.4,15.6


### **Exponentially weighted moving windows**

In [30]:
central_park_weather.assign(
    AVG = lambda x: x.TMAX.rolling('30D').mean(),
    EWMA = lambda x: x.TMAX.ewm(span=30).mean()
).loc['2018-09-29':'2018-10-08', ['TMAX', 'AVG', 'EWMA']].T

date,2018-09-29,2018-09-30,2018-10-01,2018-10-02,2018-10-03,2018-10-04,2018-10-05,2018-10-06,2018-10-07,2018-10-08
datatype,,,,,,,,,,
TMAX,22.200000,21.100000,24.400000,25.000000,23.300000,24.400000,21.700000,20.000000,26.100000,23.300000
AVG,24.723333,24.573333,24.533333,24.460000,24.163333,23.866667,23.533333,23.070000,23.143333,23.196667
EWMA,24.410887,24.197281,24.210360,24.261304,24.199285,24.212234,24.050154,23.788854,23.937960,23.896802


## **Pipes**

In [32]:
def get_info(df):
    return '%d rows, %d cols and max closing Z-score: %d' %(*df.shape, df.close.max())

In [33]:
get_info(fb_df.loc['2018-Q1'].apply(lambda x: (x - x.mean()) / x.std()))

'61 rows, 5 cols and max closing Z-score: 1'

In [34]:
fb_df.loc['2018-Q1'].apply(lambda x: (x - x.mean()) / x.std())\
    .pipe(get_info)

'61 rows, 5 cols and max closing Z-score: 1'

In [35]:
fb_df.pipe(pd.DataFrame.rolling, '20D').mean().equals(
    fb_df.rolling('20D').mean()
)

True

In [36]:
from window_calc import window_calc

In [37]:
window_calc??

Signature: window_calc(df, func, agg_dict, *args, **kwargs)
Source:   
def window_calc(df, func, agg_dict, *args, **kwargs):
    """
    Run a window calculation of your choice on a `DataFrame` object.
    
    Parameters:
        - df: The `DataFrame` object to run the calculation on.
        - func: The window calculation method that takes `df`
          as the first argument.
        - agg_dict: Information to pass to `agg()`, could be a
          dictionary mapping the columns to the aggregation
          function to use, a string name for the function,
          or the function itself.
        - args: Positional arguments to pass to `func`.
        - kwargs: Keyword arguments to pass to `func`.
    
    Returns:
        A new `DataFrame` object.
    """
    return df.pipe(func, *args, **kwargs).agg(agg_dict)
File:      ~/Desktop/DataProfessional/DataScience/Data-Science-Path/Working_with_Data/window_calc.py
Type:      function

In [40]:
window_calc(fb_df, pd.DataFrame.expanding, 'mean').head()

,open,high,low,close,volume
date,,,,,
2018-01-02,177.680000,181.5800,177.55000,181.420000,18151903.00
2018-01-03,179.780000,183.1800,179.44000,183.045000,17519233.00
2018-01-04,181.486667,184.1900,180.99320,183.473333,16306454.00
2018-01-05,182.512500,184.8675,181.97740,184.317500,15623474.25
2018-01-08,183.450000,185.6740,182.84792,185.110000,16097724.60


In [41]:
window_calc(fb_df, pd.DataFrame.ewm, 'mean', span=3).head()

,open,high,low,close,volume
date,,,,,
2018-01-02,177.680000,181.580000,177.550000,181.420000,1.815190e+07
2018-01-03,180.480000,183.713333,180.070000,183.586667,1.730834e+07
2018-01-04,183.005714,185.140000,182.372629,184.011429,1.534980e+07
2018-01-05,184.384000,186.078667,183.736560,185.525333,1.440299e+07
2018-01-08,185.837419,187.534839,185.075110,186.947097,1.625679e+07


In [42]:
window_calc(
    central_park_weather.loc['2018-10'],
    pd.DataFrame.rolling,
    {
        'TMAX': 'max', 'TMIN': 'min',
        'AWND': 'mean', 'PRCP': 'sum'
    },
    '3D'
).head()

datatype,TMAX,TMIN,AWND,PRCP
date,,,,
2018-10-01,24.4,17.2,0.900000,0.0
2018-10-02,25.0,17.2,0.900000,17.5
2018-10-03,25.0,17.2,0.966667,17.5
2018-10-04,25.0,16.1,0.800000,18.5
2018-10-05,24.4,15.6,1.033333,1.0


# **Aggregating data**

In [2]:
df_df = pd.read_csv(
    '../Datasets/fb_2018.csv', 
    index_col='date', 
    parse_dates=True
).assign(
    trading_volume = lambda x: pd.cut(
        x.volume, bins=3, labels=['low', 'med', 'high']
    )
)

weather_df = pd.read_csv(
    '../Datasets/weather_by_station.csv',
    index_col='date',
    parse_dates=True
)

In [44]:
weather_df.head()

,datatype,station,value,station_name
date,,,,
2018-01-01,PRCP,GHCND:US1CTFR0039,0.0,"STAMFORD 4.2 S, CT US"
2018-01-01,PRCP,GHCND:US1NJBG0015,0.0,"NORTH ARLINGTON 0.7 WNW, NJ US"
2018-01-01,SNOW,GHCND:US1NJBG0015,0.0,"NORTH ARLINGTON 0.7 WNW, NJ US"
2018-01-01,PRCP,GHCND:US1NJBG0017,0.0,"GLEN ROCK 0.7 SSE, NJ US"
2018-01-01,SNOW,GHCND:US1NJBG0017,0.0,"GLEN ROCK 0.7 SSE, NJ US"


In [36]:
pd.set_option('display.float_format', lambda x: '%.2f' % x)

## **Summarizing DataFrames**

In [72]:
fb_df.agg({
    'open': 'mean', 'high': 'max', 'low': 'min',
    'close': 'mean', 'volume': 'sum'
}).to_frame().T

,open,high,low,close,volume
0,171.45,218.62,123.02,171.51,6949682394.00


In [77]:
weather_df.query('station == "GHCND:USW00094728"').pivot(
    columns='datatype', values='value'
)[['SNOW', 'PRCP']].sum()

datatype
SNOW   1007.00
PRCP   1665.30
dtype: float64

In [78]:
weather_df.query('station == "GHCND:USW00094728"').pivot(
    columns='datatype', values='value'
)[['SNOW', 'PRCP']].agg('sum')

datatype
SNOW   1007.00
PRCP   1665.30
dtype: float64

In [79]:
fb_df.agg({
    'open': 'mean',
    'high': ['min', 'max'],
    'low': ['min', 'max'],
    'close': 'mean'
})

,open,high,low,close
mean,171.45,NaN,NaN,171.51
min,NaN,129.74,123.02,NaN
max,NaN,218.62,214.27,NaN


## **Aggregating by group**

In [84]:
df_df.groupby('trading_volume', observed=False).mean()

,open,high,low,close,volume
trading_volume,,,,,
low,171.36,173.46,169.31,171.43,24547207.71
med,175.82,179.42,172.11,175.14,79072559.12
high,167.73,170.48,161.57,168.16,141924023.33


In [3]:
df_df.groupby('trading_volume', observed=False)['close'].agg(['min', 'max', 'mean'])

,min,max,mean
trading_volume,,,
low,124.06,214.67,171.431771
med,152.22,217.50,175.143750
high,160.06,176.26,168.156667


In [39]:
fb_agg = df_df.groupby('trading_volume', observed=False).agg({
    'open': 'mean', 'high': ['min', 'max'],
    'low': ['min', 'max'], 'close': 'mean'
})

fb_agg

open   high           low         close
                 mean    min    max    min    max   mean
trading_volume                                          
low            171.36 129.74 216.20 123.02 212.60 171.43
med            175.82 162.85 218.62 150.75 214.27 175.14
high           167.73 161.10 180.13 149.02 173.75 168.16

In [7]:
fb_agg.loc['med', 'low']['min']

np.float64(150.75)

In [8]:
fb_agg.columns

MultiIndex([( 'open', 'mean'),
            ( 'high',  'min'),
            ( 'high',  'max'),
            (  'low',  'min'),
            (  'low',  'max'),
            ('close', 'mean')],
           )

In [40]:
fb_agg.columns = ['_'.join(col_agg) 
                  for col_agg in fb_agg.columns]

fb_agg.head()

,open_mean,high_min,high_max,low_min,low_max,close_mean
trading_volume,,,,,,
low,171.36,129.74,216.20,123.02,212.60,171.43
med,175.82,162.85,218.62,150.75,214.27,175.14
high,167.73,161.10,180.13,149.02,173.75,168.16


In [41]:
fb_agg.columns

Index(['open_mean', 'high_min', 'high_max', 'low_min', 'low_max',
       'close_mean'],
      dtype='object')

In [37]:
weather_df.loc['2018-10']\
    .query('datatype == "PRCP"')\
    .groupby(level=0)\
    .mean(numeric_only=True)\
    .head()\
    .squeeze() # squeeze turn dataframe to Series object

date
2018-10-01    0.01
2018-10-02    2.23
2018-10-03   19.69
2018-10-04    0.32
2018-10-05    0.97
Name: value, dtype: float64

In [42]:
weather_df.loc['2018-10']\
    .query('datatype == "PRCP"')\
    .groupby(level='date')\
    .mean(numeric_only=True)\
    .head()\
    .squeeze() # squeeze turn dataframe to Series object

date
2018-10-01    0.01
2018-10-02    2.23
2018-10-03   19.69
2018-10-04    0.32
2018-10-05    0.97
Name: value, dtype: float64

In [43]:
weather_df.loc['2018-10']\
    .query('datatype == "PRCP"')\
    .groupby(by='date')\
    .mean(numeric_only=True)\
    .head()\
    .squeeze() # squeeze turn dataframe to Series object

date
2018-10-01    0.01
2018-10-02    2.23
2018-10-03   19.69
2018-10-04    0.32
2018-10-05    0.97
Name: value, dtype: float64

In [52]:
weather_df.query('datatype == "PRCP"').groupby(
    ['station_name', pd.Grouper(freq='QE')]
).sum(numeric_only=True).unstack().sample(5, random_state=1)

value                                 
date                        2018-03-31 2018-06-30 2018-09-30 2018-12-31
station_name                                                           
WANTAGH 1.1 NNE, NY US          279.90     216.80     472.50     277.20
STATEN ISLAND 1.4 SE, NY US     379.40     295.30     438.80     409.90
SYOSSET 2.0 SSW, NY US          323.50     263.30     355.50     459.90
STAMFORD 4.2 S, CT US           338.00     272.10     424.70     390.00
WAYNE TWP 0.8 SSW, NJ US        246.20     295.30     620.90     422.00

> **Tip :** The **`DataFrameGroupBy`** objects returned by the **`groupby()`** method have a **`filter()`** method, which allows us to filter groups. We can use this to exclude certain groups from the aggregation. Simply pass a function that returns a Boolean for each group's subset of the dataframe (True to include the group and False to exclude it).

In [62]:
weather_df.query('datatype == "PRCP"')\
    .groupby(level=0)\
    .mean(numeric_only=True)\
    .groupby(pd.Grouper(freq='ME'))\
    .sum(numeric_only=True).value.nlargest()

date
2018-11-30   210.59
2018-09-30   193.09
2018-08-31   192.45
2018-07-31   160.98
2018-02-28   158.11
Name: value, dtype: float64

In [70]:
weather_df.query('datatype == "PRCP"')\
    .rename({'value': 'prcp'}, axis=1)\
    .groupby(level=0)\
    .mean(numeric_only=True)\
    .groupby(pd.Grouper(freq='ME'))\
    .transform('sum')['2018-01-28':'2018-02-03']

,prcp
date,
2018-01-28,69.31
2018-01-29,69.31
2018-01-30,69.31
2018-01-31,69.31
2018-02-01,158.11
2018-02-02,158.11
2018-02-03,158.11


In [73]:
weather_df.query('datatype == "PRCP"')\
    .rename(dict(value='prcp'), axis=1)\
    .groupby(level=0)\
    .mean(numeric_only=True)\
    .assign(
        total_prcp_in_month= lambda x: x.groupby(
            pd.Grouper(freq='ME')).transform('sum'),
        pct_monthly_prcp= lambda x: x.prcp.div(x.total_prcp_in_month) 
    ).nlargest(5, 'pct_monthly_prcp')

,prcp,total_prcp_in_month,pct_monthly_prcp
date,,,
2018-10-12,34.77,105.63,0.33
2018-01-13,21.66,69.31,0.31
2018-03-02,38.77,137.46,0.28
2018-04-16,39.34,140.57,0.28
2018-04-17,37.30,140.57,0.27


> **Important note :** The **`transform()`** method also works on DataFrame objects, in which case it will return a DataFrame object. We can use it to easily standardize all the columns at once.

## **Pivot tables and crosstabs**

In [79]:
df_df.pivot_table(columns='trading_volume', observed=False)

trading_volume,low,med,high
close,171.43,175.14,168.16
high,173.46,179.42,170.48
low,169.31,172.11,161.57
open,171.36,175.82,167.73
volume,24547207.71,79072559.12,141924023.33


In [84]:
weather_df\
    .reset_index()\
    .pivot_table(
        index=['date', 'station', 'station_name'],
        columns='datatype',
        values='value',
        aggfunc='median'
    ).reset_index().tail()

datatype,date,station,station_name,AWND,DAPR,MDPR,PGTM,PRCP,SNOW,SNWD,...,WSF5,WT01,WT02,WT03,WT04,WT05,WT06,WT08,WT09,WT11
28740,2018-12-31,GHCND:USW00054787,"FARMINGDALE REPUBLIC AIRPORT, NY US",5.00,NaN,NaN,2052.00,28.70,NaN,NaN,...,15.70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28741,2018-12-31,GHCND:USW00094728,"NY CITY CENTRAL PARK, NY US",NaN,NaN,NaN,NaN,25.90,0.00,0.00,...,NaN,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28742,2018-12-31,GHCND:USW00094741,"TETERBORO AIRPORT, NJ US",1.70,NaN,NaN,1954.00,29.20,NaN,NaN,...,8.90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28743,2018-12-31,GHCND:USW00094745,"WESTCHESTER CO AIRPORT, NY US",2.70,NaN,NaN,2212.00,24.40,NaN,NaN,...,11.20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28744,2018-12-31,GHCND:USW00094789,"JFK INTERNATIONAL AIRPORT, NY US",4.10,NaN,NaN,NaN,31.20,0.00,0.00,...,12.50,1.00,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [88]:
pd.crosstab(
    index=df_df.trading_volume,
    columns=df_df.index.month,
    colnames=['month'],
    values=df_df.close,
    aggfunc='median'
)

month,1,2,3,4,5,6,7,8,9,10,11,12
trading_volume,,,,,,,,,,,,
low,186.70,179.52,181.88,164.68,184.06,195.84,205.83,176.37,164.18,154.39,141.55,137.93
med,179.37,NaN,167.14,174.16,NaN,NaN,194.28,NaN,NaN,NaN,NaN,NaN
high,NaN,NaN,164.11,NaN,NaN,NaN,176.26,NaN,NaN,NaN,NaN,NaN


In [89]:
snow_data = weather_df.query('datatype == "SNOW"')

snow_data

,datatype,station,value,station_name
date,,,,
2018-01-01,SNOW,GHCND:US1NJBG0015,0.00,"NORTH ARLINGTON 0.7 WNW, NJ US"
2018-01-01,SNOW,GHCND:US1NJBG0017,0.00,"GLEN ROCK 0.7 SSE, NJ US"
2018-01-01,SNOW,GHCND:US1NJBG0018,0.00,"PALISADES PARK 0.2 WNW, NJ US"
2018-01-01,SNOW,GHCND:US1NJBG0023,0.00,"OAKLAND 0.9 SSE, NJ US"
2018-01-01,SNOW,GHCND:US1NJBG0039,0.00,"RIVER EDGE 0.4 NNE, NJ US"
...,...,...,...,...
2018-12-31,SNOW,GHCND:USC00308577,0.00,"SYOSSET, NY US"
2018-12-31,SNOW,GHCND:USW00014732,0.00,"LA GUARDIA AIRPORT, NY US"
2018-12-31,SNOW,GHCND:USW00014734,0.00,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US"


In [90]:
pd.crosstab(
    index=snow_data.station_name,
    columns=snow_data.index.month,
    colnames=['month'],
    values=snow_data.value,
    aggfunc= lambda x: (x > 0).sum(),
    margins=True, # show row and column subtotals
    margins_name='total observations of snow' # subtotals
)

month,1,2,3,4,5,6,7,8,9,10,11,12,total observations of snow
station_name,,,,,,,,,,,,,
"ALBERTSON 0.2 SSE, NY US",3.00,1.00,3.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,9
"AMITYVILLE 0.1 WSW, NY US",1.00,0.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,3
"AMITYVILLE 0.6 NNE, NY US",3.00,1.00,3.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,8
"ARMONK 0.3 SE, NY US",6.00,4.00,6.00,3.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,3.00,23
"BLOOMINGDALE 0.7 SSE, NJ US",2.00,1.00,3.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...
"WESTFIELD 0.6 NE, NJ US",3.00,0.00,4.00,1.00,0.00,NaN,0.00,0.00,0.00,NaN,1.00,NaN,9
"WOODBRIDGE TWP 1.1 ESE, NJ US",4.00,1.00,3.00,2.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,11
"WOODBRIDGE TWP 1.1 NNE, NJ US",2.00,1.00,3.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,7


# **Working with time series data**

In [91]:
fb_df = pd.read_csv(
    '../Datasets/fb_2018.csv',
    index_col='date',
    parse_dates=True
).assign(
    trading_volume= lambda x: pd.cut(
        x.volume, bins=3, labels=['low', 'med', 'high']
    )
)

fb_df.head()

,open,high,low,close,volume,trading_volume
date,,,,,,
2018-01-02,177.68,181.58,177.55,181.42,18151903,low
2018-01-03,181.88,184.78,181.33,184.67,16886563,low
2018-01-04,184.90,186.21,184.10,184.33,13880896,low
2018-01-05,185.59,186.90,184.93,186.85,13574535,low
2018-01-08,187.20,188.90,186.33,188.28,17994726,low


## **Time-based selection and filtering**

In [92]:
fb_df['2018-10-11':'2018-10-15']

,open,high,low,close,volume,trading_volume
date,,,,,,
2018-10-11,150.13,154.81,149.16,153.35,35338901,low
2018-10-12,156.73,156.89,151.30,153.74,25293492,low
2018-10-15,153.32,155.57,152.55,153.52,15433521,low


In [98]:
fb_df.loc['2018-Q1'].equals(fb_df['2018-01':'2018-03'])

True

In [101]:
fb_df.first('1W')

/tmp/ipykernel_4092/1615129269.py:1: FutureWarning: first is deprecated and will be removed in a future version. Please create a mask and filter using `.loc` instead
  fb_df.first('1W')


,open,high,low,close,volume,trading_volume
date,,,,,,
2018-01-02,177.68,181.58,177.55,181.42,18151903,low
2018-01-03,181.88,184.78,181.33,184.67,16886563,low
2018-01-04,184.90,186.21,184.10,184.33,13880896,low
2018-01-05,185.59,186.90,184.93,186.85,13574535,low


In [107]:
fb_df['2018-01-01':'2018-01-07']

,open,high,low,close,volume,trading_volume
date,,,,,,
2018-01-02,177.68,181.58,177.55,181.42,18151903,low
2018-01-03,181.88,184.78,181.33,184.67,16886563,low
2018-01-04,184.90,186.21,184.10,184.33,13880896,low
2018-01-05,185.59,186.90,184.93,186.85,13574535,low


In [108]:
fb_df.last('1W')

/tmp/ipykernel_4092/939349685.py:1: FutureWarning: last is deprecated and will be removed in a future version. Please create a mask and filter using `.loc` instead
  fb_df.last('1W')


,open,high,low,close,volume,trading_volume
date,,,,,,
2018-12-31,134.45,134.64,129.95,131.09,24625308,low


In [110]:
fb_reindex = fb_df.reindex(
    pd.date_range('2018-01-01', '2018-12-31', freq='D')
)

In [113]:
fb_reindex.first('1D').isna().squeeze().all()

/tmp/ipykernel_4092/949518457.py:1: FutureWarning: first is deprecated and will be removed in a future version. Please create a mask and filter using `.loc` instead
  fb_reindex.first('1D').isna().squeeze().all()


np.True_

In [114]:
fb_reindex.loc['2018-Q1'].first_valid_index()

Timestamp('2018-01-02 00:00:00')

In [115]:
fb_reindex.loc['2018-Q1'].last_valid_index()

Timestamp('2018-03-29 00:00:00')

In [116]:
fb_reindex.asof('2018-03-31')

open                  155.15
high                  161.42
low                   154.14
close                 159.79
volume           59434293.00
trading_volume           low
Name: 2018-03-31 00:00:00, dtype: object

In [121]:
fb_reindex.loc['2018-03'].tail().iloc[2]

open                  155.15
high                  161.42
low                   154.14
close                 159.79
volume           59434293.00
trading_volume           low
Name: 2018-03-29 00:00:00, dtype: object

In [125]:
stock_data_per_minute = pd.read_csv(
    '../Datasets/fb_week_of_may_20_per_minute.csv',
    index_col='date',
    parse_dates=True,
    date_format='%Y-%m-%d %H-%M',
    # date_parser= lambda x: pd.to_datetime(x, format='%Y-%m-%d %H-%M') Canot use both date_format and date_parser
)

stock_data_per_minute.head()

,open,high,low,close,volume
date,,,,,
2019-05-20 09:30:00,181.62,181.62,181.62,181.62,159049.00
2019-05-20 09:31:00,182.61,182.61,182.61,182.61,468017.00
2019-05-20 09:32:00,182.75,182.75,182.75,182.75,97258.00
2019-05-20 09:33:00,182.95,182.95,182.95,182.95,43961.00
2019-05-20 09:34:00,183.06,183.06,183.06,183.06,79562.00


In [126]:
stock_data_per_minute.groupby(pd.Grouper(freq='1D')).agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum'
})

,open,high,low,close,volume
date,,,,,
2019-05-20,181.62,184.18,181.62,182.72,10044838.00
2019-05-21,184.53,185.58,183.97,184.82,7198405.00
2019-05-22,184.81,186.56,184.01,185.32,8412433.00
2019-05-23,182.50,183.73,179.76,180.87,12479171.00
2019-05-24,182.33,183.52,181.04,181.06,7686030.00


In [128]:
stock_data_per_minute.at_time('09:30')

,open,high,low,close,volume
date,,,,,
2019-05-20 09:30:00,181.62,181.62,181.62,181.62,159049.00
2019-05-21 09:30:00,184.53,184.53,184.53,184.53,58171.00
2019-05-22 09:30:00,184.81,184.81,184.81,184.81,41585.00
2019-05-23 09:30:00,182.50,182.50,182.50,182.50,121930.00
2019-05-24 09:30:00,182.33,182.33,182.33,182.33,52681.00


In [131]:
stock_data_per_minute.between_time('15:59', '16:00')

,open,high,low,close,volume
date,,,,,
2019-05-20 15:59:00,182.91,182.91,182.91,182.91,134569.00
2019-05-20 16:00:00,182.72,182.72,182.72,182.72,1113672.00
2019-05-21 15:59:00,184.84,184.84,184.84,184.84,61606.00
2019-05-21 16:00:00,184.82,184.82,184.82,184.82,801080.00
2019-05-22 15:59:00,185.29,185.29,185.29,185.29,96099.00
2019-05-22 16:00:00,185.32,185.32,185.32,185.32,1220993.00
2019-05-23 15:59:00,180.72,180.72,180.72,180.72,109648.00
2019-05-23 16:00:00,180.87,180.87,180.87,180.87,1329217.00
2019-05-24 15:59:00,181.07,181.07,181.07,181.07,52994.00


In [143]:
shares_traded_in_first_30_min = stock_data_per_minute\
    .between_time('09:30', '10:00')\
    .groupby(pd.Grouper(freq='1D'))\
    .filter(lambda x: (x.volume > 0).all())\
    .volume.mean()

shares_traded_in_first_30_min

np.float64(64934.25806451613)

In [144]:
shares_traded_in_last_30_min = stock_data_per_minute\
    .between_time('15:30', '16:00')\
    .groupby(pd.Grouper(freq='1D'))\
    .filter(lambda x: (x.volume > 0).all())\
    .volume.mean()

shares_traded_in_last_30_min

np.float64(46341.290322580644)

In [145]:
shares_traded_in_first_30_min - shares_traded_in_last_30_min

np.float64(18592.967741935485)

> **Tip :** We can use the **`normalize()`** method on **`DatetimeIndex`** objects or after first accessing the dt attribute of a Series object to normalize all the datetimes to midnight. This is helpful when the time isn't adding value to our data.

## **Shifting for lagged data**

In [146]:
fb_df.assign(
    prior_close= lambda x: x.close.shift(),
    after_hours_change_in_price= lambda x: x.open - x.prior_close,
    abs_change= lambda x: x.after_hours_change_in_price.abs()
).nlargest(5, 'abs_change')

,open,high,low,close,volume,trading_volume,prior_close,after_hours_change_in_price,abs_change
date,,,,,,,,,
2018-07-26,174.89,180.13,173.75,176.26,169803668,high,217.50,-42.61,42.61
2018-04-26,173.22,176.27,170.80,174.16,77556934,med,159.69,13.53,13.53
2018-01-12,178.06,181.48,177.40,179.37,77551299,med,187.77,-9.71,9.71
2018-10-31,155.00,156.40,148.96,151.79,60101251,low,146.22,8.78,8.78
2018-03-19,177.01,177.17,170.06,172.56,88140060,med,185.09,-8.08,8.08


In [148]:
fb_df['2018-01-10':'2018-01-12']

,open,high,low,close,volume,trading_volume
date,,,,,,
2018-01-10,186.94,187.89,185.63,187.84,10529894,low
2018-01-11,188.40,188.40,187.38,187.77,9588587,low
2018-01-12,178.06,181.48,177.40,179.37,77551299,med


> **Tip :** To add/subtract time from the datetimes in the index, consider using **`Timedelta`** objects instead.

## **Differenced data**

In [151]:
(fb_df.drop(columns='trading_volume') - fb_df.drop(columns='trading_volume').shift())\
    .equals(fb_df.drop(columns='trading_volume').diff())

True

In [153]:
fb_df.drop(columns='trading_volume').diff().head()

,open,high,low,close,volume
date,,,,,
2018-01-02,NaN,NaN,NaN,NaN,NaN
2018-01-03,4.20,3.20,3.78,3.25,-1265340.00
2018-01-04,3.02,1.43,2.77,-0.34,-3005667.00
2018-01-05,0.69,0.69,0.83,2.52,-306361.00
2018-01-08,1.61,2.00,1.40,1.43,4420191.00


> **Tip :** To specify the number of periods that are used for the difference, simply pass in an integer to **`diff()`**. Note that this number can be negative.

## **Resampling**

In [157]:
stock_data_per_minute.resample('1D').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum'
})

,open,high,low,close,volume
date,,,,,
2019-05-20,181.62,184.18,181.62,182.72,10044838.00
2019-05-21,184.53,185.58,183.97,184.82,7198405.00
2019-05-22,184.81,186.56,184.01,185.32,8412433.00
2019-05-23,182.50,183.73,179.76,180.87,12479171.00
2019-05-24,182.33,183.52,181.04,181.06,7686030.00


In [160]:
fb_df.resample('QE').mean(numeric_only=True)

,open,high,low,close,volume
date,,,,,
2018-03-31,179.47,181.79,177.04,179.55,32926396.70
2018-06-30,180.37,182.28,178.60,180.70,24055317.75
2018-09-30,180.81,182.89,178.96,181.03,27019824.76
2018-12-31,145.27,147.62,142.72,144.87,26974331.73


In [172]:
fb_df.drop(columns='trading_volume').resample('QE').apply(
    lambda x: x.last('1D').values - x.first('1D').values
).to_frame()

/tmp/ipykernel_4092/1262211835.py:2: FutureWarning: last is deprecated and will be removed in a future version. Please create a mask and filter using `.loc` instead
  lambda x: x.last('1D').values - x.first('1D').values
/tmp/ipykernel_4092/1262211835.py:2: FutureWarning: first is deprecated and will be removed in a future version. Please create a mask and filter using `.loc` instead
  lambda x: x.last('1D').values - x.first('1D').values
/tmp/ipykernel_4092/1262211835.py:2: FutureWarning: last is deprecated and will be removed in a future version. Please create a mask and filter using `.loc` instead
  lambda x: x.last('1D').values - x.first('1D').values
/tmp/ipykernel_4092/1262211835.py:2: FutureWarning: first is deprecated and will be removed in a future version. Please create a mask and filter using `.loc` instead
  lambda x: x.last('1D').values - x.first('1D').values


,0
date,
2018-03-31,"[[-22.53, -20.160000000000025, -23.41000000000..."
2018-06-30,"[[39.50999999999999, 38.399700000000024, 39.84..."
2018-09-30,"[[-25.039999999999992, -28.659999999999997, -2..."
2018-12-31,"[[-28.580000000000013, -31.24000000000001, -31..."


In [174]:
melted_stock_data = pd.read_csv(
    '../Datasets/melted_stock_data.csv',
    index_col='date',
    parse_dates=True
)

melted_stock_data.head()

,price
date,
2019-05-20 09:30:00,181.62
2019-05-20 09:31:00,182.61
2019-05-20 09:32:00,182.75
2019-05-20 09:33:00,182.95
2019-05-20 09:34:00,183.06


In [177]:
melted_stock_data.resample('1D').ohlc()['price']

,open,high,low,close
date,,,,
2019-05-20,181.62,184.18,181.62,182.72
2019-05-21,184.53,185.58,183.97,184.82
2019-05-22,184.81,186.56,184.01,185.32
2019-05-23,182.50,183.73,179.76,180.87
2019-05-24,182.33,183.52,181.04,181.06


In [183]:
fb_df.resample('6h').asfreq().head()

,open,high,low,close,volume,trading_volume
date,,,,,,
2018-01-02 00:00:00,177.68,181.58,177.55,181.42,18151903.00,low
2018-01-02 06:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-02 12:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-02 18:00:00,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03 00:00:00,181.88,184.78,181.33,184.67,16886563.00,low


- The following are a few ways we can handle the NaN values. In the interest of brevity, examples of these are in the notebook:
    - Use **`pad()`** after resample() to forward fill.
    - Call **`fillna()`** after **`resample()`**, when we handled missing values.
    - Use **`asfreq()`** followed by **`assign()`** to handle each column individually.

## **Merging time series**

In [184]:
import sqlite3

In [190]:
with sqlite3.connect('../Datasets/stocks.db') as connection:
    fb_prices = pd.read_sql(
        'SELECT * FROM fb_prices',
        connection,
        index_col='date',
        parse_dates=['date']
    )
    aapl_prices = pd.read_sql(
        'SELECT * FROM aapl_prices',
        connection,
        index_col='date',
        parse_dates=['date']
    )

In [191]:
fb_prices.head()

,FB
date,
2019-05-20 09:30:00,181.62
2019-05-20 09:31:00,182.61
2019-05-20 09:32:00,182.75
2019-05-20 09:33:00,182.95
2019-05-20 09:34:00,183.06


In [192]:
aapl_prices.head()

,AAPL
date,
2019-05-20 09:30:00,183.52
2019-05-20 09:31:52,182.87
2019-05-20 09:32:36,182.50
2019-05-20 09:33:34,182.11
2019-05-20 09:34:55,181.50


In [194]:
fb_prices.index.second.unique()

Index([0], dtype='int32', name='date')

In [196]:
aapl_prices.index.second.unique()

Index([ 0, 52, 36, 34, 55, 35,  7, 12, 59, 17,  5, 20, 26, 23, 54, 49, 19, 53,
       11, 22, 13, 21, 10, 46, 42, 38, 33, 18, 16,  9, 56, 39,  2, 50, 31, 58,
       48, 24, 29,  6, 47, 51, 40,  3, 15, 14, 25,  4, 43,  8, 32, 27, 30, 45,
        1, 44, 57, 41, 37, 28],
      dtype='int32', name='date')

In [198]:
pd.merge_asof(
    fb_prices, 
    aapl_prices, 
    left_index=True, 
    right_index=True,
    direction='nearest', # merge with nearest minute
    tolerance=pd.Timedelta(30, unit='s')
).head()

,FB,AAPL
date,,
2019-05-20 09:30:00,181.62,183.52
2019-05-20 09:31:00,182.61,NaN
2019-05-20 09:32:00,182.75,182.87
2019-05-20 09:33:00,182.95,182.50
2019-05-20 09:34:00,183.06,182.11


In [199]:
pd.merge_ordered(
    fb_prices.reset_index(), 
    aapl_prices.reset_index(),
).set_index('date').head()

,FB,AAPL
date,,
2019-05-20 09:30:00,181.62,183.52
2019-05-20 09:31:00,182.61,NaN
2019-05-20 09:31:52,NaN,182.87
2019-05-20 09:32:00,182.75,NaN
2019-05-20 09:32:36,NaN,182.50


> **Tip :** We can pass **`fill_method='ffill'`** to **`pd.merge_ordered()`** to forward-fill the first NaN after a value, but it does not propagate beyond that; alternatively, we can chain a call to **`fillna()`**.

# **Further reading**

- Check out the following resources for more information on the topics that were covered in this chapter:
    - **Intro to SQL:** Querying and managing data: https://www.khanacademy.org/computing/computer-programming/sql
    - **(Pandas) Comparison with SQL:** https://pandas.pydata.org/pandas-docs/stable/getting_started/comparison/comparison_with_sql.html
    - **Set Operations:** https://www.probabilitycourse.com/chapter1/1_2_2_set_operations.php
    - **\*args and \*\*kwargs in Python explained:** https://pythontips.com/2013/08/04/args-and-kwargs-in-python-explained/

# **Practice**

In [3]:
df = pd.read_csv('../Datasets/earthquakes.csv')

df.head()

,alert,cdi,code,detail,dmin,felt,gap,ids,mag,magType,...,sources,status,time,title,tsunami,type,types,tz,updated,url
0,NaN,NaN,37389218,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.008693,NaN,85.0,",ci37389218,",1.35,ml,...,",ci,",automatic,1539475168010,"M 1.4 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.0,1539475395144,https://earthquake.usgs.gov/earthquakes/eventp...
1,NaN,NaN,37389202,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.020030,NaN,79.0,",ci37389202,",1.29,ml,...,",ci,",automatic,1539475129610,"M 1.3 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.0,1539475253925,https://earthquake.usgs.gov/earthquakes/eventp...
2,NaN,4.4,37389194,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.021370,28.0,21.0,",ci37389194,",3.42,ml,...,",ci,",automatic,1539475062610,"M 3.4 - 8km NE of Aguanga, CA",0,earthquake,",dyfi,focal-mechanism,geoserve,nearby-cities,o...",-480.0,1539536756176,https://earthquake.usgs.gov/earthquakes/eventp...
3,NaN,NaN,37389186,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.026180,NaN,39.0,",ci37389186,",0.44,ml,...,",ci,",automatic,1539474978070,"M 0.4 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.0,1539475196167,https://earthquake.usgs.gov/earthquakes/eventp...
4,NaN,NaN,73096941,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.077990,NaN,192.0,",nc73096941,",2.16,md,...,",nc,",automatic,1539474716050,"M 2.2 - 10km NW of Avenal, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,scit...",-480.0,1539477547926,https://earthquake.usgs.gov/earthquakes/eventp...


### **With the earthquakes.csv file, select all the earthquakes in Japan with a magnitude of 4.9 or greater using the mb magnitude type.**

In [8]:
df.query(
    'type == "earthquake" and title.str.contains("Japan") and mag >= 4.9 and magType == "mb"'
)

,alert,cdi,code,detail,dmin,felt,gap,ids,mag,magType,...,sources,status,time,title,tsunami,type,types,tz,updated,url
1563,NaN,NaN,1000h8mj,https://earthquake.usgs.gov/fdsnws/event/1/que...,3.773,NaN,101.0,",us1000h8mj,",4.9,mb,...,",us,",reviewed,1538977532250,"M 4.9 - 293km ESE of Iwo Jima, Japan",0,earthquake,",geoserve,origin,phase-data,",600.0,1538978864040,https://earthquake.usgs.gov/earthquakes/eventp...
2576,NaN,3.1,1000h7cn,https://earthquake.usgs.gov/fdsnws/event/1/que...,1.015,7.0,33.0,",us1000h7cn,",5.4,mb,...,",us,",reviewed,1538697528010,"M 5.4 - 37km E of Tomakomai, Japan",0,earthquake,",dyfi,geoserve,moment-tensor,origin,phase-data,",540.0,1538757701040,https://earthquake.usgs.gov/earthquakes/eventp...
3072,NaN,3.1,1000h6ac,https://earthquake.usgs.gov/fdsnws/event/1/que...,2.082,15.0,118.0,",us1000h6ac,",4.9,mb,...,",us,",reviewed,1538579732490,"M 4.9 - 15km ENE of Hasaki, Japan",0,earthquake,",dyfi,geoserve,origin,phase-data,",540.0,1539230846942,https://earthquake.usgs.gov/earthquakes/eventp...
3632,NaN,3.1,1000h5gh,https://earthquake.usgs.gov/fdsnws/event/1/que...,1.449,19.0,117.0,",us1000h5gh,",4.9,mb,...,",us,",reviewed,1538450871260,"M 4.9 - 53km ESE of Hitachi, Japan",0,earthquake,",dyfi,geoserve,origin,phase-data,",540.0,1538581746455,https://earthquake.usgs.gov/earthquakes/eventp...


In [10]:
df[
    (df['type'] == "earthquake") & (df['title'].str.contains("Japan") & (df["mag"] >= 4.9) & (df["magType"] == "mb")) 
]

,alert,cdi,code,detail,dmin,felt,gap,ids,mag,magType,...,sources,status,time,title,tsunami,type,types,tz,updated,url
1563,NaN,NaN,1000h8mj,https://earthquake.usgs.gov/fdsnws/event/1/que...,3.773,NaN,101.0,",us1000h8mj,",4.9,mb,...,",us,",reviewed,1538977532250,"M 4.9 - 293km ESE of Iwo Jima, Japan",0,earthquake,",geoserve,origin,phase-data,",600.0,1538978864040,https://earthquake.usgs.gov/earthquakes/eventp...
2576,NaN,3.1,1000h7cn,https://earthquake.usgs.gov/fdsnws/event/1/que...,1.015,7.0,33.0,",us1000h7cn,",5.4,mb,...,",us,",reviewed,1538697528010,"M 5.4 - 37km E of Tomakomai, Japan",0,earthquake,",dyfi,geoserve,moment-tensor,origin,phase-data,",540.0,1538757701040,https://earthquake.usgs.gov/earthquakes/eventp...
3072,NaN,3.1,1000h6ac,https://earthquake.usgs.gov/fdsnws/event/1/que...,2.082,15.0,118.0,",us1000h6ac,",4.9,mb,...,",us,",reviewed,1538579732490,"M 4.9 - 15km ENE of Hasaki, Japan",0,earthquake,",dyfi,geoserve,origin,phase-data,",540.0,1539230846942,https://earthquake.usgs.gov/earthquakes/eventp...
3632,NaN,3.1,1000h5gh,https://earthquake.usgs.gov/fdsnws/event/1/que...,1.449,19.0,117.0,",us1000h5gh,",4.9,mb,...,",us,",reviewed,1538450871260,"M 4.9 - 53km ESE of Hitachi, Japan",0,earthquake,",dyfi,geoserve,origin,phase-data,",540.0,1538581746455,https://earthquake.usgs.gov/earthquakes/eventp...


In [11]:
df.query(
    'type == "earthquake" and title.str.contains("Japan") and mag >= 4.9 and magType == "mb"'
).equals(df[
    (df['type'] == "earthquake") & (df['title'].str.contains("Japan") & (df["mag"] >= 4.9) & (df["magType"] == "mb")) 
])

True

### **Create bins for each full number of earthquake magnitude (for instance, the first bin is (0, 1], the second is (1, 2], and so on) with the ml magnitude type and count how many are in each bin.**

In [38]:
df.assign(
    magnitude_cuted = pd.cut(
        df.mag, 
        bins=[0, 1, 2, 3, 4, 5, 6, 7]
    )
)[df.magType == "ml"].groupby(
    by="magnitude_cuted", observed=False
).agg({
    "magnitude_cuted" : "count"
})

,magnitude_cuted
magnitude_cuted,
"(0, 1]",2207
"(1, 2]",3105
"(2, 3]",862
"(3, 4]",122
"(4, 5]",2
"(5, 6]",1
"(6, 7]",0


### **3. Using the faang.csv file, group by the ticker and resample to monthly frequency. Make the following aggregations:**

In [2]:
faang_df = pd.read_csv('./faang.csv', index_col="date", parse_dates=True)

faang_df.head()

,high,low,open,close,volume,ticker
date,,,,,,
2018-01-02,43.075001,42.314999,42.540001,43.064999,102223600.0,AAPL
2018-01-03,43.637501,42.990002,43.132500,43.057499,118071600.0,AAPL
2018-01-04,43.367500,43.020000,43.134998,43.257500,89738400.0,AAPL
2018-01-05,43.842499,43.262501,43.360001,43.750000,94640000.0,AAPL
2018-01-08,43.902500,43.482498,43.587502,43.587502,82271200.0,AAPL


#### **a) Mean of the opening price**
#### **b) Maximum of the high price**
#### **c) Minimum of the low price**
#### **d) Mean of the closing price**
#### **e) Sum of the volume traded**

In [62]:
pd.set_option("display.float_format", lambda x: "%.2f" % x)

In [63]:
faang_df.groupby(by='ticker').resample('ME').agg({
    "open": "mean",
    "high": "max",
    "low": "min",
    "close": "mean",
    "volume": "sum"
})

open    high     low   close        volume
ticker date                                                    
AAPL   2018-01-31   43.51   45.03   41.17   43.50 2638717600.00
       2018-02-28   41.82   45.15   37.56   41.91 3711577200.00
       2018-03-31   43.76   45.88   41.24   43.62 2854910800.00
       2018-04-30   42.44   44.74   40.16   42.46 2664617200.00
       2018-05-31   46.24   47.59   41.32   46.38 2483905200.00
       2018-06-30   47.18   48.55   45.18   47.16 2110498000.00
       2018-07-31   47.55   48.99   45.85   47.58 1574765600.00
       2018-08-31   53.12   57.22   49.33   53.34 2801275600.00
       2018-09-30   55.58   57.42   53.83   55.52 2715888000.00
       2018-10-31   55.30   58.37   51.52   55.21 3158994000.00
       2018-11-30   47.95   55.59   42.56   47.81 3845305600.00
       2018-12-31   41.31   46.24   36.65   41.07 3595690000.00
AMZN   2018-01-31 1301.38 1472.58 1170.51 1309.01   96371200.00
       2018-02-28 1447.11 1528.70 1265.93 1442.36  137784000.00
       2018-03-31 1542.16 1617.54 1365.20 1540.37  130400100.00
       2018-04-30 1475.84 1638.10 1352.88 1468.22  129919600.00
       2018-05-31 1590.47 1635.00 1546.02 1594.90   71615500.00
       2018-06-30 1699.09 1763.10 1635.09 1698.82   85941300.00
       2018-07-31 1786.31 1880.05 1678.06 1784.65   97521100.00
       2018-08-31 1891.96 2025.57 1776.02 1897.85   96575800.00
       2018-09-30 1969.24 2050.50 1865.00 1966.08   94445500.00
       2018-10-31 1799.63 2033.19 1476.36 1782.06  183220800.00
       2018-11-30 1622.32 1784.00 1420.00 1625.48  139290000.00
       2018-12-31 1572.92 1778.34 1307.00 1559.44  154812700.00
FB     2018-01-31  184.58  190.66  175.80  184.96  495655700.00
       2018-02-28  180.72  195.32  167.18  180.27  516251600.00
       2018-03-31  173.45  186.10  149.02  173.49  996201700.00
       2018-04-30  164.16  177.10  150.51  163.81  750072700.00
       2018-05-31  181.91  192.72  170.23  182.93  401144100.00
       2018-06-30  194.97  203.55  186.43  195.27  387265600.00
       2018-07-31  199.33  218.62  166.56  199.97  647030700.00
       2018-08-31  177.60  188.30  170.27  177.49  548832700.00
       2018-09-30  164.23  173.89  158.87  164.38  500468800.00
       2018-10-31  154.87  165.88  139.03  154.19  622446300.00
       2018-11-30  141.76  154.13  126.85  141.64  518151700.00
       2018-12-31  137.53  147.19  123.02  137.16  558786200.00
GOOG   2018-01-31 1127.20 1186.89 1045.23 1130.77   28738400.00
       2018-02-28 1088.63 1174.00  992.56 1088.21   42382000.00
       2018-03-31 1096.11 1177.05  980.64 1091.49   45353300.00
       2018-04-30 1038.42 1094.17  990.37 1035.70   41715900.00
       2018-05-31 1064.02 1110.75 1006.29 1069.28   31849400.00
       2018-06-30 1136.40 1186.29 1096.01 1137.63   32096000.00
       2018-07-31 1183.46 1273.89 1093.80 1187.59   31940100.00
       2018-08-31 1226.16 1256.50 1188.24 1225.67   28808400.00
       2018-09-30 1176.88 1212.99 1146.91 1175.81   28862400.00
       2018-10-31 1116.08 1209.96  995.83 1110.94   48494700.00
       2018-11-30 1054.97 1095.57  996.02 1056.16   36735100.00
       2018-12-31 1042.62 1124.65  970.11 1037.42   40257600.00
NFLX   2018-01-31  231.27  286.81  195.42  232.91  238377600.00
       2018-02-28  270.87  297.36  236.11  271.44  184585800.00
       2018-03-31  312.71  333.98  275.90  312.23  263449400.00
       2018-04-30  309.13  338.82  271.22  307.47  262006000.00
       2018-05-31  329.78  356.10  305.73  331.54  142050800.00
       2018-06-30  384.56  423.21  352.82  384.13  244031800.00
       2018-07-31  380.97  419.77  328.00  381.52  305393800.00
       2018-08-31  345.41  376.81  310.93  346.26  213122300.00
       2018-09-30  363.33  383.20  335.83  362.64  170832100.00
       2018-10-31  340.03  386.80  271.21  335.45  363589800.00
       2018-11-30  290.64  332.05  250.00  290.34  257126400.00
       2018-12-31  266.31  298.72  231.23  265.30  234310000.00

### **4. Build a crosstab with the earthquake data between the tsunami column and the magType column. Rather than showing the frequency count, show the maximum magnitude that was observed for each combination. Put the magnitude type along the columns.**

In [88]:
pd.crosstab(
    index=df.index,
    columns=[df.tsunami, df.magType, df.mag],
    colnames=["Tsunami", "magnitude Type", "magnitude"],
    values=df["mag"],
    aggfunc="max"
)

Tsunami           0                                               ...    1  \
magnitude Type   mb                                               ...  mww   
magnitude      3.60 3.80 3.90 4.00 4.10 4.20 4.30 4.40 4.50 4.60  ... 5.40   
row_0                                                             ...        
0               NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
1               NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
2               NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
3               NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
4               NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
...             ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...  ...   
9327            NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
9328            NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
9329            NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
9330            NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   
9331            NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  ...  NaN   

Tsunami                                                      
magnitude Type                                               
magnitude      5.60 5.90 6.00 6.10 6.20 6.50 6.70 7.00 7.50  
row_0                                                        
0               NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
1               NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
2               NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
3               NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
4               NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
...             ...  ...  ...  ...  ...  ...  ...  ...  ...  
9327            NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
9328            NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
9329            NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
9330            NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
9331            NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  

[9331 rows x 852 columns]

### **5. Calculate the rolling 60-day aggregations of the OHLC data by ticker for the FAANG data. Use the same aggregations as exercise 3.**

In [90]:
faang_df.groupby(by="ticker").rolling('60D').agg({
    "open": "mean",
    "high": "max",
    "low": "min",
    "close": "mean",
    "volume": "sum"
})

open   high    low  close       volume
ticker date                                               
AAPL   2018-01-02  42.54  43.08  42.31  43.06 102223600.00
       2018-01-03  42.84  43.64  42.31  43.06 220295200.00
       2018-01-04  42.94  43.64  42.31  43.13 310033600.00
       2018-01-05  43.04  43.84  42.31  43.28 404673600.00
       2018-01-08  43.15  43.90  42.31  43.34 486944800.00
...                  ...    ...    ...    ...          ...
NFLX   2018-12-24 283.51 332.05 233.68 281.93 525657600.00
       2018-12-26 281.84 332.05 231.23 280.78 520444300.00
       2018-12-27 281.07 332.05 231.23 280.16 532679500.00
       2018-12-28 279.92 332.05 231.23 279.46 521973500.00
       2018-12-31 278.43 332.05 231.23 277.45 476314900.00

[1255 rows x 5 columns]

### **6. Create a pivot table of the FAANG data that compares the stocks. Put the ticker in the rows and show the averages of the OHLC and volume traded data.**

In [93]:
faang_df.pivot_table(
    index="ticker",
    aggfunc="mean"
)

,close,high,low,open,volume
ticker,,,,,
AAPL,47.26,47.75,46.80,47.28,136080258.17
AMZN,1641.73,1662.84,1619.84,1644.07,5648994.42
FB,171.51,173.61,169.30,171.47,27658596.81
GOOG,1113.23,1125.78,1101.00,1113.55,1741965.34
NFLX,319.29,325.22,313.19,319.62,11469624.70


### **7. Calculate the Z-scores for each numeric column of Amazon's data (ticker is AMZN) in Q4 2018 using apply().**

In [118]:
faang_df.query('ticker == "AMZN"')\
    .loc["2018-Q4", ["close", "high", "low", "open", "volume"]]\
    .apply(lambda x: x.sub(x.mean()).div(x.std()))

,close,high,low,open,volume
date,,,,,
2018-10-01,2.39,2.37,2.50,2.34,-1.63
2018-10-02,2.16,2.23,2.25,2.19,-0.86
2018-10-03,2.03,2.06,2.14,2.07,-0.92
2018-10-04,1.72,1.82,1.78,1.85,-0.13
2018-10-05,1.58,1.63,1.55,1.64,-0.30
...,...,...,...,...,...
2018-12-24,-2.23,-2.16,-2.19,-2.18,-0.14
2018-12-26,-1.34,-1.61,-1.81,-2.03,1.12
2018-12-27,-1.40,-1.64,-1.63,-1.46,0.85


### **8. Add event descriptions:**

#### **a) Create a dataframe with the following three columns: ticker, date, and event. The columns should have the following values:**
    - i) ticker: 'FB'
    - ii) date: ['2018-07-25', '2018-03-19', '2018-03-20']
    - iii) event: ['Disappointing user growth announced after close.', 'Cambridge Analytica story', 'FTC investigation']

In [5]:
fb_df = pd.DataFrame({
    'ticker': "FB",
    'date': pd.to_datetime(['2018-07-25', '2018-03-19', '2018-03-20']),
    'event': ['Disappointing user growth announced after close.', 'Cambridge Analytica story', 'FTC investigation']
})

fb_df

,ticker,date,event
0,FB,2018-07-25,Disappointing user growth announced after close.
1,FB,2018-03-19,Cambridge Analytica story
2,FB,2018-03-20,FTC investigation


In [7]:
fb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   ticker  3 non-null      object        
 1   date    3 non-null      datetime64[ns]
 2   event   3 non-null      object        
dtypes: datetime64[ns](1), object(2)
memory usage: 204.0+ bytes


#### **b) Set the index to ['date', 'ticker'].**

In [10]:
fb_df.set_index(['date', 'ticker'], inplace=True)

In [11]:
fb_df

,,event
date,ticker,
2018-07-25,FB,Disappointing user growth announced after close.
2018-03-19,FB,Cambridge Analytica story
2018-03-20,FB,FTC investigation


#### **c) Merge this data with the FAANG data using an outer join**

In [19]:
faang_df.merge(
    fb_df,
    how='outer',
    on=['date', 'ticker']
)

,high,low,open,close,volume,ticker,event
date,,,,,,,
2018-01-02,43.075001,42.314999,42.540001,43.064999,102223600.0,AAPL,NaN
2018-01-02,1190.000000,1170.510010,1172.000000,1189.010010,2694500.0,AMZN,NaN
2018-01-02,181.580002,177.550003,177.679993,181.419998,18151900.0,FB,NaN
2018-01-02,1066.939941,1045.229980,1048.339966,1065.000000,1237600.0,GOOG,NaN
2018-01-02,201.649994,195.419998,196.100006,201.070007,10966900.0,NFLX,NaN
...,...,...,...,...,...,...,...
2018-12-31,39.840000,39.119999,39.632500,39.435001,140014000.0,AAPL,NaN
2018-12-31,1520.760010,1487.000000,1510.800049,1501.969971,6954500.0,AMZN,NaN
2018-12-31,134.639999,129.949997,134.449997,131.089996,24625300.0,FB,NaN


### **9. Use the `transform()` method on the FAANG data to represent all the values in terms of the first date in the data. To do so, divide all the values for each ticker by the values for the first date in the data for that ticker. This is referred to as an index, and the data for the first date is the base (https://ec.europa.eu/eurostat/statistics-explained/index.php/Beginners:Statistical_concept_-_Index_and_base_year). When data is in this format, we can easily see growth over time. Hint: `transform()` can take a function name.**

In [49]:
faang_df.groupby(by='ticker').transform(lambda x: x.first('1D').div(x))

/tmp/ipykernel_3925/2082845228.py:1: FutureWarning: first is deprecated and will be removed in a future version. Please create a mask and filter using `.loc` instead
  faang_df.groupby(by='ticker').transform(lambda x: x.first('1D').div(x))


,high,low,open,close,volume
date,,,,,
2018-01-02,1.0,1.0,1.0,1.0,1.0
2018-01-03,NaN,NaN,NaN,NaN,NaN
2018-01-04,NaN,NaN,NaN,NaN,NaN
2018-01-05,NaN,NaN,NaN,NaN,NaN
2018-01-08,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...
2018-12-24,NaN,NaN,NaN,NaN,NaN
2018-12-26,NaN,NaN,NaN,NaN,NaN
2018-12-27,NaN,NaN,NaN,NaN,NaN


### **10. The European Centre for Disease Prevention and Control (ECDC) provides an open dataset on COVID-19 cases called daily number of new reported cases of COVID-19 by country worldwide (https://www.ecdc.europa.eu/en/publications-data/download-todays-data-geographic-distribution-covid-19-cases-worldwide). This dataset is updated daily, but we will use a snapshot that contains data through September 18, 2020.**

#### **a) Prepare the data:**
    - i) Read in the data in the covid19_cases.csv file.
    - ii) Create a date column by parsing the dateRep column into a datetime.
    - iii) Set the date column as the index.
    - iv) Use the replace() method to update all occurrences of United_States_of_America and United_Kingdom to USA and UK, respectively.
    - v) Sort the index.

In [175]:
covid_df = pd.read_csv('../Datasets/covid19_cases.csv', index_col='dateRep', parse_dates=True)\
    .replace(['United_States_of_America', 'United_Kingdom'], ['USA', 'UK'])\

covid_df

,day,month,year,cases,deaths,countriesAndTerritories,geoId,countryterritoryCode,popData2019,continentExp,Cumulative_number_for_14_days_of_COVID-19_cases_per_100000
dateRep,,,,,,,,,,,
01/01/2020,1,1,2020,0,0,Lithuania,LT,LTU,2794184.0,Europe,NaN
01/01/2020,1,1,2020,0,0,Iceland,IS,ISL,356991.0,Europe,NaN
01/01/2020,1,1,2020,0,0,Nepal,NP,NPL,28608715.0,Asia,NaN
01/01/2020,1,1,2020,0,0,San_Marino,SM,SMR,34453.0,Europe,NaN
01/01/2020,1,1,2020,0,0,Canada,CA,CAN,37411038.0,America,NaN
...,...,...,...,...,...,...,...,...,...,...,...
18/09/2020,18,9,2020,822,2,Denmark,DK,DNK,5806081.0,Europe,69.220529
18/09/2020,18,9,2020,4326,84,Iraq,IQ,IRQ,39309789.0,Asia,153.513925
18/09/2020,18,9,2020,90,0,Bahamas,BS,BHS,389486.0,America,203.088173


In [178]:
covid_df.index = pd.to_datetime(covid_df.index, format='mixed')

covid_df.index

DatetimeIndex(['2020-01-01', '2020-01-01', '2020-01-01', '2020-01-01',
               '2020-01-01', '2020-01-01', '2020-01-01', '2020-01-01',
               '2020-01-01', '2020-01-01',
               ...
               '2020-09-18', '2020-09-18', '2020-09-18', '2020-09-18',
               '2020-09-18', '2020-09-18', '2020-09-18', '2020-09-18',
               '2020-09-18', '2020-09-18'],
              dtype='datetime64[ns]', name='dateRep', length=43443, freq=None)

### **b) For the five countries with the most cases (cumulative), find the day with the largest number of cases.**

In [180]:
covid_df[covid_df['Cumulative_number_for_14_days_of_COVID-19_cases_per_100000'].isin(covid_df.groupby(by='countriesAndTerritories').agg({
            'Cumulative_number_for_14_days_of_COVID-19_cases_per_100000': 'max',
        }).nlargest(5, 'Cumulative_number_for_14_days_of_COVID-19_cases_per_100000').values.reshape(5))]

,day,month,year,cases,deaths,countriesAndTerritories,geoId,countryterritoryCode,popData2019,continentExp,Cumulative_number_for_14_days_of_COVID-19_cases_per_100000
dateRep,,,,,,,,,,,
2020-09-04,9,4,2020,1,0,Holy_See,VA,VAT,815.0,Europe,858.895706
2020-05-06,5,6,2020,1581,0,Qatar,QA,QAT,2832071.0,Asia,885.924117
2020-08-20,20,8,2020,175,1,Aruba,AW,ABW,106310.0,America,1058.225943
2020-05-09,5,9,2020,22,1,Turks_and_Caicos_islands,TC,TCA,38194.0,America,602.188825
2020-09-18,18,9,2020,5165,4,Israel,IL,ISR,8519373.0,Asia,606.535246


### **c) Find the 7-day average change in COVID-19 cases for the last week in the data for the five countries with the most cases.**

In [204]:
covid_df.last('1W').groupby(by='countriesAndTerritories').agg({
    'cases': ['max', 'mean']
}).nlargest(5, ('cases', 'max'))

/tmp/ipykernel_3925/3550353236.py:1: FutureWarning: last is deprecated and will be removed in a future version. Please create a mask and filter using `.loc` instead
  covid_df.last('1W').groupby(by='countriesAndTerritories').agg({


cases              
                           max          mean
countriesAndTerritories                     
India                    97894  93838.777778
USA                      51473  38518.333333
Brazil                   43718  32590.333333
Spain                    27404  10740.111111
Argentina                12259  11222.777778

### **d) Find the first date that each country other than China had cases.**

In [209]:
covid_df.first('1D').query('cases > 0 and countriesAndTerritories != "China"')

/tmp/ipykernel_3925/1966433232.py:1: FutureWarning: first is deprecated and will be removed in a future version. Please create a mask and filter using `.loc` instead
  covid_df.first('1D').query('cases > 0 and countriesAndTerritories != "China"')


,day,month,year,cases,deaths,countriesAndTerritories,geoId,countryterritoryCode,popData2019,continentExp,Cumulative_number_for_14_days_of_COVID-19_cases_per_100000
dateRep,,,,,,,,,,,


### **e) Rank the countries by cumulative cases using percentiles.**

In [210]:
covid_df.assign(
    rank = lambda x: x["Cumulative_number_for_14_days_of_COVID-19_cases_per_100000"].rank()
)

,day,month,year,cases,deaths,countriesAndTerritories,geoId,countryterritoryCode,popData2019,continentExp,Cumulative_number_for_14_days_of_COVID-19_cases_per_100000,rank
dateRep,,,,,,,,,,,,
2020-01-01,1,1,2020,0,0,Lithuania,LT,LTU,2794184.0,Europe,NaN,NaN
2020-01-01,1,1,2020,0,0,Iceland,IS,ISL,356991.0,Europe,NaN,NaN
2020-01-01,1,1,2020,0,0,Nepal,NP,NPL,28608715.0,Asia,NaN,NaN
2020-01-01,1,1,2020,0,0,San_Marino,SM,SMR,34453.0,Europe,NaN,NaN
2020-01-01,1,1,2020,0,0,Canada,CA,CAN,37411038.0,America,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2020-09-18,18,9,2020,822,2,Denmark,DK,DNK,5806081.0,Europe,69.220529,35038.0
2020-09-18,18,9,2020,4326,84,Iraq,IQ,IRQ,39309789.0,Asia,153.513925,38426.0
2020-09-18,18,9,2020,90,0,Bahamas,BS,BHS,389486.0,America,203.088173,39133.5
